# Stanford NLP (Stanza) → Multilingual

- Collection of NLP tools developed by Stanford NLP group.
- Before DL became dominant, Stanford NLP was giving state-of-the-art performance on tasks like:
  - Tokenization
  - POS tagging
  - etc.
- All in one package.

---

# Stanford NLP Pipeline

```text
Raw text
   ↓
Sentence split
   ↓
Tokenization
   ↓
POS tagging
   ↓
Lemmatization
   ↓
NER
   ↓
Parsing
   ↓
Coreference resolution

In [1]:
# Install Java
!apt-get install -y openjdk-11-jdk

# Download Stanford CoreNLP
!wget http://nlp.stanford.edu/software/stanford-corenlp-4.5.5.zip
!unzip stanford-corenlp-4.5.5.zip

# Download English model
!wget http://nlp.stanford.edu/software/stanford-corenlp-4.5.5-models-english.jar -P stanford-corenlp-4.5.5/

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  at-spi2-core fonts-dejavu-core fonts-dejavu-extra gsettings-desktop-schemas
  libatk-bridge2.0-0 libatk-wrapper-java libatk-wrapper-java-jni libatk1.0-0
  libatk1.0-data libatspi2.0-0 libxcomposite1 libxt-dev libxtst6 libxxf86dga1
  openjdk-11-jdk-headless openjdk-11-jre openjdk-11-jre-headless
  session-migration x11-utils
Suggested packages:
  libxt-doc openjdk-11-demo openjdk-11-source visualvm libnss-mdns
  fonts-ipafont-gothic fonts-ipafont-mincho fonts-wqy-microhei
  | fonts-wqy-zenhei fonts-indic mesa-utils
The following NEW packages will be installed:
  at-spi2-core fonts-dejavu-core fonts-dejavu-extra gsettings-desktop-schemas
  libatk-bridge2.0-0 libatk-wrapper-java libatk-wrapper-java-jni libatk1.0-0
  libatk1.0-data libatspi2.0-0 libxcomposite1 libxt-dev libxtst6 libxxf86dga1
  openjdk-11-jdk openjdk-11-jdk-headless openjdk-

In [2]:
!pip install stanza

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 46.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 773.7/773.7 kB 40.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 36.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 418.7/418.7 kB 25.4 MB/s eta 0:00:00


In [3]:
import stanza

# Download and initialize English pipeline
stanza.download('en')
nlp = stanza.Pipeline(lang='en', processors='tokenize,mwt,pos,lemma,depparse,ner,constituency')

INFO:stanza:Downloaded file to /root/.cache/stanza/1.12.0/resources/resources.json
INFO:stanza:Downloading default packages for language: en (English) ...


INFO:stanza:Downloaded file to /root/.cache/stanza/1.12.0/resources/en/default.zip
INFO:stanza:Finished downloading models and saved to /root/.cache/stanza/1.12.0/resources
INFO:stanza:Checking for updates to resources.json in case models have been updated.  Note: this behavior can be turned off with download_method=None or download_method=DownloadMethod.REUSE_RESOURCES


INFO:stanza:Downloaded file to /root/.cache/stanza/1.12.0/resources/resources.json
INFO:stanza:Loading these models for language: en (English):
| Processor    | Package                   |
--------------------------------------------
| tokenize     | combined                  |
| mwt          | combined                  |
| pos          | combined_charlm           |
| lemma        | combined_nocharlm         |
| constituency | ptb3-revised_charlm       |
| depparse     | combined_charlm           |
| ner          | ontonotes-ww-multi_charlm |

INFO:stanza:Using device: cpu
INFO:stanza:Loading: tokenize
INFO:stanza:Loading: mwt
INFO:stanza:Loading: pos
INFO:stanza:Loading: lemma
INFO:stanza:Loading: constituency
INFO:stanza:Loading: depparse
INFO:stanza:Loading: ner
INFO:stanza:Done loading processors!


## Event Extraction

In [4]:
text = "John bought a new laptop from Amazon."
doc = nlp(text)

for sent in doc.sentences:
    for word in sent.words:
        if word.upos in ['VERB', 'AUX']:
            print(f"Event: {word.text}")
            # Print arguments (subject, object, etc.)
            for dep in sent.words:
                if dep.head == word.id:
                    print(f"  Argument ({dep.deprel}): {dep.text}")

Event: bought
  Argument (nsubj): John
  Argument (obj): laptop
  Argument (obl): Amazon
  Argument (punct): .


## Temporal Information Extraction

In [5]:
text = "The meeting was held on March 3rd, 2024 at 10 AM."
doc = nlp(text)

for ent in doc.ents:
    if ent.type == 'DATE' or ent.type == 'TIME':
        print(f"Temporal expression: {ent.text} -> {ent.type}")

Temporal expression: March 3rd, 2024 -> DATE
Temporal expression: 10 AM -> TIME


## Question Answering Pipelines (Syntax-aware)

In [6]:
question = "Who bought the laptop?"
context = "John bought a new laptop from Amazon."

doc = nlp(context)
# Use dependency parsing to extract subject/object
for sent in doc.sentences:
    for word in sent.words:
        if word.deprel == 'nsubj':
            print(f"Possible Answer: {word.text}")

Possible Answer: John


## Structure-Aware Text Summarization (basic compression)

In [7]:
text = "The manager, who was extremely busy, scheduled a meeting with the team."
doc = nlp(text)

for sent in doc.sentences:
    essential = [word.text for word in sent.words if word.deprel in ['nsubj', 'obj', 'root']]
    print("Compressed sentence:", ' '.join(essential))


Compressed sentence: manager who scheduled meeting


## Multilingual Document Analysis

In [8]:
# Hindi
stanza.download('hi')
hi_nlp = stanza.Pipeline(lang='hi')

text = "राम ने सीता के लिए फूल खरीदे।"
doc = hi_nlp(text)
for sent in doc.sentences:
    for word in sent.words:
        print(f"{word.text} ({word.upos}) - {word.deprel} -> {word.head}")


INFO:stanza:Downloaded file to /root/.cache/stanza/1.12.0/resources/resources.json
INFO:stanza:Downloading default packages for language: hi (Hindi) ...


INFO:stanza:Downloaded file to /root/.cache/stanza/1.12.0/resources/hi/default.zip
INFO:stanza:Finished downloading models and saved to /root/.cache/stanza/1.12.0/resources
INFO:stanza:Checking for updates to resources.json in case models have been updated.  Note: this behavior can be turned off with download_method=None or download_method=DownloadMethod.REUSE_RESOURCES


INFO:stanza:Downloaded file to /root/.cache/stanza/1.12.0/resources/resources.json
INFO:stanza:Loading these models for language: hi (Hindi):
| Processor | Package       |
-----------------------------
| tokenize  | hdtb          |
| pos       | hdtb_charlm   |
| lemma     | hdtb_nocharlm |
| depparse  | hdtb_charlm   |
| ner       | ilner_charlm  |

INFO:stanza:Using device: cpu
INFO:stanza:Loading: tokenize
INFO:stanza:Loading: pos
INFO:stanza:Loading: lemma
INFO:stanza:Loading: depparse
INFO:stanza:Loading: ner
INFO:stanza:Done loading processors!


राम (PROPN) - nsubj -> 7
ने (ADP) - case -> 1
सीता (PROPN) - obl -> 7
के (ADP) - case -> 3
लिए (ADP) - case -> 3
फूल (NOUN) - obj -> 7
खरीदे (VERB) - root -> 0
। (PUNCT) - punct -> 7


## Textual Entailment Features (syntactic/semantic features)


In [9]:
text1 = "The cat sat on the mat."
text2 = "A feline was sitting on a rug."

doc1 = nlp(text1)
doc2 = nlp(text2)

def get_dependencies(doc):
    return set((w.text, w.deprel) for s in doc.sentences for w in s.words)

print("Overlap in syntactic roles:", get_dependencies(doc1) & get_dependencies(doc2))


Overlap in syntactic roles: {('.', 'punct'), ('on', 'case')}


## Named Entity Normalization / Linking

In [10]:
text = "Apple was founded by Steve Jobs in Cupertino."

doc = nlp(text)
for ent in doc.ents:
    print(f"Entity: {ent.text} ({ent.type})")
    # Placeholder: Entity linking to KB could go here (e.g., Wikidata, DBPedia)

Entity: Apple (ORG)
Entity: Steve Jobs (ORG)
Entity: Cupertino (GPE)


## Syntax-aware Machine Translation Support

In [11]:
text = "The dog chased the cat under the table."
doc = nlp(text)

for sent in doc.sentences:
    print("Tree Structure:")
    print(sent.constituency)
    # Tree structure can be passed as features to a syntax-aware MT model

Tree Structure:
(ROOT (S (NP (DT The) (NN dog)) (VP (VBD chased) (NP (DT the) (NN cat)) (PP (IN under) (NP (DT the) (NN table)))) (. .)))


## Knowledge Graph Triple Extraction

In [12]:
text = "Elon Musk founded SpaceX and Tesla."
doc = nlp(text)

for sent in doc.sentences:
    subj, obj, rel = None, None, None
    for word in sent.words:
        if word.deprel == 'nsubj':
            subj = word.text
        elif word.deprel == 'obj':
            obj = word.text
        elif word.upos == 'VERB':
            rel = word.text
    if subj and obj and rel:
        print(f"Triple: ({subj}, {rel}, {obj})")

Triple: (Elon, founded, SpaceX)


# Stanford Stanza

| Point | Description |
|---|---|
| Library type | Modern Python library |
| Developed by | Stanford |
| Purpose | Used for NLP tasks |

---

# Comparison

| Feature | spaCy | Stanza | NLTK | TextBlob |
|---|---|---|---|---|
| Speed | Very fast | Slower | Slower | Fast |
| Accuracy | Very good | Very good | Moderate | Moderate |
| Ease of use | Moderate | Moderate | Very easy | Super easy |
| Best for | Production | Research | Teaching | Prototype |
| Installation | Easy | Medium / Java | Easy | Easy |
| Coreference resolution | ❌ | ✅ | ❌ | ❌ |
| Dependency parsing | ✅ | ✅ | — | ✅ |
| Multilingual support | 64+ | 66+ | Partial | Few |
| Pre-trained DL models | Many | Many | ❌ | ❌ |
| Super built-in sentiment | ❌ | ✅ / two | ❌ | ✅ |